In [9]:
# 1. IMPORTS
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [10]:
# ==============================
# 2. LOAD DATA
# ==============================
df1 = pd.read_parquet("yellow_tripdata_2025-01.parquet")
df2 = pd.read_parquet("yellow_tripdata_2025-02.parquet")
df3 = pd.read_parquet("yellow_tripdata_2025-03.parquet")

df = pd.concat([df1, df2, df3], ignore_index=True)

# ==============================

In [11]:
# 3. CLEAN DATA
# ==============================
df['pickup_time'] = pd.to_datetime(df['tpep_pickup_datetime'])

df = df[
    (df['trip_distance'] > 0) &
    (df['fare_amount'] > 0) &
    (df['PULocationID'].notna())
]

df = df[['pickup_time', 'PULocationID', 'trip_distance', 'fare_amount']]


In [12]:
# 4. BUILD DEMAND DATASET
# ==============================
df.set_index('pickup_time', inplace=True)

demand_df = df.groupby([
    pd.Grouper(freq='15min'),
    'PULocationID'
]).agg({
    'fare_amount': 'mean',
    'trip_distance': 'mean'
}).rename(columns={
    'fare_amount': 'avg_fare',
    'trip_distance': 'avg_distance'
})

demand_df['demand'] = df.groupby([
    pd.Grouper(freq='15min'),
    'PULocationID'
]).size()

demand_df = demand_df.reset_index()


In [13]:
# ==============================
# 5. FEATURE ENGINEERING
# ==============================
demand_df['hour'] = demand_df['pickup_time'].dt.hour
demand_df['dayofweek'] = demand_df['pickup_time'].dt.dayofweek

# Cyclical features
demand_df['hour_sin'] = np.sin(2*np.pi*demand_df['hour']/24)
demand_df['hour_cos'] = np.cos(2*np.pi*demand_df['hour']/24)

demand_df['day_sin'] = np.sin(2*np.pi*demand_df['dayofweek']/7)
demand_df['day_cos'] = np.cos(2*np.pi*demand_df['dayofweek']/7)

# Sort for time series
demand_df = demand_df.sort_values(['PULocationID', 'pickup_time'])

# Lag features
for lag in [1, 2, 4, 8]:
    demand_df[f'lag_{lag}'] = demand_df.groupby('PULocationID')['demand'].shift(lag)

# Rolling features
demand_df['rolling_mean'] = demand_df.groupby('PULocationID')['demand'].rolling(4).mean().reset_index(0, drop=True)
demand_df['rolling_std'] = demand_df.groupby('PULocationID')['demand'].rolling(4).std().reset_index(0, drop=True)

# Momentum
demand_df['momentum'] = demand_df['lag_1'] - demand_df['lag_2']

# Driver density (proxy)
demand_df['driver_density'] = demand_df.groupby('PULocationID')['demand'].rolling(2).mean().reset_index(0, drop=True)

demand_df = demand_df.dropna()


In [14]:
# ==============================
# 6. TRAIN / TEST SPLIT
# ==============================
train = demand_df[demand_df['pickup_time'] < '2025-03-01']
test = demand_df[demand_df['pickup_time'] >= '2025-03-01']

features = [
    'hour_sin', 'hour_cos',
    'day_sin', 'day_cos',
    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    'rolling_mean', 'rolling_std',
    'momentum',
    'driver_density',
    'PULocationID'
]

X_train = train[features]
y_train = train['demand']

X_test = test[features]
y_test = test['demand']


In [15]:

# ==============================
# 7. TRAIN MODEL
# ==============================
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [16]:
# ==============================
# 8. EVALUATION
# ==============================
def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-6))) * 100

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAPE:", MAPE(y_test, y_pred))


MAE: 0.28921523690223694
RMSE: 1.137717840274547
MAPE: 4.064022661656365


In [17]:
# ==============================
# 9. DRIVER SIMULATION
# ==============================
current_time = test['pickup_time'].min()

current_zone = test[
    test['pickup_time'] == current_time
]['PULocationID'].mode()[0]

print("Driver starting zone:", current_zone)

current_time_data = test[test['pickup_time'] == current_time].copy()


Driver starting zone: 4


In [18]:
# ==============================
# 10. PREDICT DEMAND FOR CURRENT TIME
# ==============================
current_time_data['predicted_demand'] = model.predict(current_time_data[features])

# ==============================
# 11. WSM FUNCTION (IMPROVED)
# ==============================
def compute_wsm(row, current_zone):
    alpha, beta, gamma, delta = 1.0, 0.5, 0.3, 0.7

    R = row['predicted_demand'] * row['avg_fare'] / (1 + row['driver_density'])
    I = 1 / (row['predicted_demand'] + 1e-6)
    C = abs(row['PULocationID'] - current_zone)
    D = row['driver_density']

    return alpha*R - beta*I - gamma*C - delta*D

current_time_data['WSM'] = current_time_data.apply(
    lambda row: compute_wsm(row, current_zone), axis=1
)


In [19]:
# ==============================
# 12. RECOMMENDED ZONE
# ==============================
best_zone_row = current_time_data.loc[current_time_data['WSM'].idxmax()]

recommended_zone = best_zone_row['PULocationID']

print("\n✅ Recommended Zone:", recommended_zone)
print("WSM Score:", best_zone_row['WSM'])

# ==============================


✅ Recommended Zone: 17
WSM Score: 30.432310094628832


In [20]:
# 13. BASELINES
# ==============================
# Stay baseline
stay_zone = current_zone

# Max demand baseline (real-time)
max_demand_zone = current_time_data.loc[
    current_time_data['predicted_demand'].idxmax(), 'PULocationID'
]

print("\nBaseline - Stay Zone:", stay_zone)
print("Baseline - Max Demand Zone:", max_demand_zone)



Baseline - Stay Zone: 4
Baseline - Max Demand Zone: 79
